# HRM-Text: fine-tuning para raciocínio hierárquico em TypeScript

Notebook para Google Colab: carrega um dataset de instrução/código, prepara exemplos no formato PrefixLM do HRM-Text, aplica LoRA quando compatível, treina, avalia amostras e exporta adaptadores/checkpoints.

**Importante:** fine-tuning ensina padrões de resolução estruturada; não garante raciocínio correto. O HRM-Text-1B usa uma arquitetura própria e exige compatibilidade entre Transformers, PEFT e `token_type_ids`. O notebook valida isso e falha com diagnóstico em vez de fingir que treinou. Para uma GPU T4, comece com `MAX_LENGTH=512`, `MAX_SAMPLES=1500` e batch pequeno.

In [ ]:
import os, sys, json, random, subprocess
from pathlib import Path

SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Instala versões atuais compatíveis; se houver conflito com o modelo, reinicie o runtime e execute novamente.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'datasets', 'accelerate', 'peft', 'bitsandbytes', 'trl', 'evaluate', 'sentencepiece'])

In [ ]:
import torch, transformers, datasets, peft
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| Transformers:', transformers.__version__, '| PEFT:', peft.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print('GPU não encontrada. No Colab, selecione Ambiente de execução > Alterar tipo de ambiente de execução > GPU.')

## 1. Configuração

O modelo HRM-Text-1B é o padrão. Se a versão de Transformers/PEFT não aceitar a arquitetura ou os campos do modelo, veja a célula de diagnóstico antes de trocar para outro modelo causal: isso muda a arquitetura treinada, não é mais fine-tuning do HRM.

In [ ]:
MODEL_ID = 'sapientinc/HRM-Text-1B'
DATASET_ID = 'grenishrai/typescript-dataset'
DATASET_CONFIG = None       # use string se o dataset exigir uma configuração
TRAIN_SPLIT = 'train'
MAX_SAMPLES = 1500           # None para usar todos; ajuste à GPU/tempo
MAX_LENGTH = 512
VALIDATION_SIZE = 0.05
OUTPUT_DIR = '/content/hrm-ts-lora'
SEED = 42

# Treino conservador para Colab T4. Aumente gradualmente após validar a primeira execução.
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
USE_4BIT = True


## 2. Carregar e inspecionar o dataset

A célula seguinte não presume que o dataset tenha exatamente as colunas `instruction` e `output`: ela mostra as colunas e tenta identificar campos comuns. Revise os exemplos antes de treinar.

In [ ]:
from datasets import load_dataset

kwargs = {'split': TRAIN_SPLIT}
if DATASET_CONFIG:
    raw_ds = load_dataset(DATASET_ID, DATASET_CONFIG, **kwargs)
else:
    raw_ds = load_dataset(DATASET_ID, **kwargs)
print(raw_ds)
print('Colunas:', raw_ds.column_names)
print('Exemplo:', raw_ds[0])

In [ ]:
def first_present(row, keys):
    for key in keys:
        value = row.get(key)
        if value is not None and str(value).strip():
            return str(value).strip()
    return ''

INSTRUCTION_KEYS = ['instruction', 'prompt', 'question', 'problem', 'task', 'input']
ANSWER_KEYS = ['output', 'response', 'answer', 'completion', 'solution', 'code', 'target']
TEXT_KEYS = ['text', 'content']

def normalize_row(row):
    instruction = first_present(row, INSTRUCTION_KEYS)
    answer = first_present(row, ANSWER_KEYS)
    if not instruction and not answer:
        text = first_present(row, TEXT_KEYS)
        if text:
            # Evita inventar uma separação pergunta/resposta para texto bruto.
            instruction, answer = 'Analise o seguinte material de TypeScript e produza uma resposta útil:', text
    elif instruction and not answer:
        # Se input e output não existirem, não converte silenciosamente um prompt em alvo.
        answer = first_present(row, TEXT_KEYS)
    if not instruction or not answer:
        return None
    return {'instruction': instruction, 'answer': answer}

normalized = [normalize_row(x) for x in raw_ds]
normalized = [x for x in normalized if x and len(x['instruction']) > 2 and len(x['answer']) > 1]
if not normalized:
    raise ValueError('Não consegui mapear pares instrução/resposta. Inspecione as colunas e ajuste INSTRUCTION_KEYS / ANSWER_KEYS.')
from datasets import Dataset
all_ds = Dataset.from_list(normalized)
if MAX_SAMPLES:
    all_ds = all_ds.shuffle(seed=SEED).select(range(min(MAX_SAMPLES, len(all_ds))))
split = all_ds.train_test_split(test_size=VALIDATION_SIZE, seed=SEED)
train_ds, eval_ds = split['train'], split['test']
print('Treino:', len(train_ds), '| Validação:', len(eval_ds))
print(train_ds[0])

## 3. Tokenização e máscara PrefixLM

O prompt recebe `token_type_ids=1` e o alvo recebe `0`; a loss é calculada apenas nos tokens da resposta. O exemplo é descartado se a resposta ficar vazia após truncamento. Os tokens de condição são os descritos no cartão do HRM-Text.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if TOKENIZER.pad_token is None:
    TOKENIZER.pad_token = TOKENIZER.eos_token or TOKENIZER.unk_token

CONDITION = '<|quad_end|><|object_ref_end|>'

def tokenize_example(example):
    prompt = f'<|im_start|>{CONDITION}Instrução: {example["instruction"]}\nResposta:'
    answer = example['answer'] + (TOKENIZER.eos_token or '')
    p_ids = TOKENIZER(prompt, add_special_tokens=False)['input_ids']
    a_ids = TOKENIZER(answer, add_special_tokens=False)['input_ids']
    # Reserva espaço para a resposta, sem ultrapassar MAX_LENGTH.
    if len(a_ids) >= MAX_LENGTH:
        a_ids = a_ids[:MAX_LENGTH]
        p_ids = []
    else:
        p_ids = p_ids[-(MAX_LENGTH - len(a_ids)):]
    input_ids = p_ids + a_ids
    if not a_ids:
        return {'input_ids': [], 'attention_mask': [], 'token_type_ids': [], 'labels': []}
    return {
        'input_ids': input_ids,
        'attention_mask': [1] * len(input_ids),
        'token_type_ids': [1] * len(p_ids) + [0] * len(a_ids),
        'labels': [-100] * len(p_ids) + a_ids,
    }

train_tok = train_ds.map(tokenize_example, remove_columns=train_ds.column_names)
eval_tok = eval_ds.map(tokenize_example, remove_columns=eval_ds.column_names)
train_tok = train_tok.filter(lambda x: len(x['input_ids']) > 0 and any(v != -100 for v in x['labels']))
eval_tok = eval_tok.filter(lambda x: len(x['input_ids']) > 0 and any(v != -100 for v in x['labels']))
print('Exemplo tokenizado:', {k: len(v) for k,v in train_tok[0].items()})
print('Tokens supervisionados:', sum(x != -100 for x in train_tok[0]['labels']))

## 4. Carregar modelo e configurar LoRA

O carregamento em 4 bits é opcional e depende de GPU CUDA. LoRA é aplicado às camadas lineares detectadas automaticamente. Se a arquitetura customizada não funcionar com PEFT ou rejeitar `token_type_ids`, pare e consulte a mensagem: não remova os campos PrefixLM sem entender a alteração de objetivo.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig
import inspect

quant_config = None
if USE_4BIT and torch.cuda.is_available():
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

load_kwargs = {'trust_remote_code': True, 'low_cpu_mem_usage': True}
if quant_config:
    load_kwargs.update({'quantization_config': quant_config, 'device_map': 'auto'})
else:
    load_kwargs['torch_dtype'] = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
print('Classe:', type(model).__name__)
print('Forward aceita token_type_ids?', 'token_type_ids' in inspect.signature(model.forward).parameters or any(p.kind == inspect.Parameter.VAR_KEYWORD for p in inspect.signature(model.forward).parameters.values()))

if quant_config:
    model = prepare_model_for_kbit_training(model)

# Descobre projeções lineares elegíveis; exclui cabeças de saída e embeddings para limitar os adaptadores.
linear_names = []
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear) and not any(x in name.lower() for x in ['lm_head', 'embed', 'classifier']):
        leaf = name.split('.')[-1]
        if leaf not in linear_names:
            linear_names.append(leaf)
print('Nomes de módulos lineares candidatos:', linear_names[:80])
if not linear_names:
    raise RuntimeError('Não encontrei camadas torch.nn.Linear para LoRA. Esta arquitetura requer configuração de módulos específica.')

# Seleciona projeções de atenção/MLP quando os nomes usuais existem; senão usa os nomes lineares encontrados.
preferred = [n for n in linear_names if any(k in n.lower() for k in ['q_proj','k_proj','v_proj','o_proj','query','key','value','dense','gate_proj','up_proj','down_proj'])]
target_modules = preferred or linear_names
lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias='none', task_type='CAUSAL_LM', target_modules=target_modules)
try:
    model = get_peft_model(model, lora_config)
except Exception as e:
    raise RuntimeError(f'PEFT/LoRA não foi compatível com esta arquitetura HRM. Erro original: {type(e).__name__}: {e}. Verifique o código de treinamento oficial do HRM-Text antes de adaptar módulos.') from e
model.print_trainable_parameters()

## 5. Collator e treinamento

Usamos padding dinâmico e preservamos `token_type_ids`. Se o `Trainer`/modelo não aceitar esse argumento, o treinamento deve falhar explicitamente. Para a arquitetura HRM, compare os resultados com o script de treinamento oficial antes de interpretar o checkpoint.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

class PrefixLMCollator:
    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of
    def __call__(self, features):
        max_len = max(len(f['input_ids']) for f in features)
        if self.pad_to_multiple_of:
            max_len = ((max_len + self.pad_to_multiple_of - 1)//self.pad_to_multiple_of)*self.pad_to_multiple_of
        pad_id = self.tokenizer.pad_token_id
        batch = {'input_ids': [], 'attention_mask': [], 'token_type_ids': [], 'labels': []}
        for f in features:
            n = max_len - len(f['input_ids'])
            batch['input_ids'].append(f['input_ids'] + [pad_id]*n)
            batch['attention_mask'].append(f['attention_mask'] + [0]*n)
            batch['token_type_ids'].append(f['token_type_ids'] + [0]*n)
            batch['labels'].append(f['labels'] + [-100]*n)
        return {k: torch.tensor(v, dtype=torch.long) for k,v in batch.items()}

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    bf16=False,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit' if quant_config else 'adamw_torch',
    report_to='none',
    seed=SEED,
    data_seed=SEED,
    remove_unused_columns=False,
    label_names=['labels'],
)
trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=eval_tok, data_collator=PrefixLMCollator(TOKENIZER))
print('Passos estimados:', max(1, (len(train_tok) + MICRO_BATCH_SIZE*GRADIENT_ACCUMULATION_STEPS - 1)//(MICRO_BATCH_SIZE*GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS)


In [ ]:
train_result = trainer.train()
print(train_result)
eval_result = trainer.evaluate()
print('Avaliação:', eval_result)

## 6. Inferência de teste

A geração usa o prefixo PrefixLM. Avalie a qualidade com casos que não aparecem no treino; a loss de validação sozinha não prova capacidade de raciocínio.

In [ ]:
def generate_answer(question, max_new_tokens=256):
    prompt = f'<|im_start|>{CONDITION}Instrução: {question}\nResposta:'
    inputs = TOKENIZER(prompt, return_tensors='pt', add_special_tokens=False)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k,v in inputs.items()}
    inputs['token_type_ids'] = torch.ones_like(inputs['input_ids'])
    model.eval()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=TOKENIZER.pad_token_id, eos_token_id=TOKENIZER.eos_token_id)
    return TOKENIZER.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)

print(generate_answer('Escreva uma função TypeScript genérica que retorne o primeiro elemento de um array ou undefined quando estiver vazio.'))

In [ ]:
print('Exemplos de validação:')
for i in range(min(3, len(eval_ds))):
    row = eval_ds[i]
    print('\nINSTRUÇÃO:', row['instruction'][:500])
    print('REFERÊNCIA:', row['answer'][:700])
    print('MODELO:', generate_answer(row['instruction'], max_new_tokens=180)[:700])

## 7. Salvar adaptador e tokenizer

O artefato salvo é um adaptador LoRA, não um modelo completo independente. Para inferência, carregue o modelo base compatível e aplique o adaptador.

In [ ]:
from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
TOKENIZER.save_pretrained(OUTPUT_DIR)
print('Artefatos:', OUTPUT_DIR)
print('Arquivos:', sorted(p.name for p in Path(OUTPUT_DIR).iterdir())[:40])

# Compacta para download opcional.
import shutil
archive = shutil.make_archive('/content/hrm-ts-lora', 'zip', OUTPUT_DIR)
print('ZIP:', archive)

## 8. Checklist de validação

- [ ] O runtime está usando GPU e não ocorreu OOM.
- [ ] As colunas do dataset foram mapeadas corretamente e as respostas são de qualidade.
- [ ] `token_type_ids` e a máscara PrefixLM são aceitos pela implementação carregada.
- [ ] A loss de treino e validação foi registrada; exemplos fora do treino foram inspecionados.
- [ ] O checkpoint foi recarregado e testado em uma sessão nova.
- [ ] Licenças do modelo e dataset foram verificadas antes de redistribuir pesos ou dados.

**Limitações:** o dataset recomendado pode conter ruído ou esquema diferente do esperado; confira as primeiras amostras. O modelo base HRM-Text foi pré-treinado predominantemente em inglês e sem dados de código segundo seu cartão, então desempenho em TypeScript/português precisa ser medido. O notebook usa LoRA genérico e não substitui o pipeline oficial de pretraining/treino HRM.